In [ ]:
# based on calibration script from TP 8

from tdmclient import ClientAsync
from threading import Timer
import numpy as np

client = ClientAsync()
node = await client.wait_for_node()
await node.lock()

# taken from Stack Overflow discussion used in TP 8: https://stackoverflow.com/questions/3393612/run-certain-code-every-n-seconds
class RepeatedTimer(object):
    def __init__(self, interval, function, *args, **kwargs):
        self._timer     = None
        self.interval   = interval
        self.function   = function
        self.args       = args
        self.kwargs     = kwargs
        self.is_running = False
        self.start()

    def _run(self):
        self.is_running = False
        self.start()
        self.function(*self.args, **self.kwargs)

    def start(self):
        if not self.is_running:
            self._timer = Timer(self.interval, self._run)
            self._timer.start()
            self.is_running = True

    def stop(self):
        self._timer.cancel()
        self.is_running = False

raw_speed = 100 #50 100
Ts = 0.1
nsteps = 200 # 100
distance_mm = 667 # 184 #330
thymio_data = []

def motor_speeds(left, right):
    return {
        "motor.left.target": [left],
        "motor.right.target": [right]
    }

def get_data():
    thymio_data.append({"left_speed":node["motor.left.speed"],
                    "right_speed":node["motor.right.speed"]})

await node.wait_for_variables()
rt = RepeatedTimer(Ts, get_data)
try:
    await client.sleep(5)
    node.send_set_variables(motor_speeds(raw_speed, raw_speed))
    await client.sleep(nsteps*Ts)
finally:
    rt.stop()
    node.send_set_variables(motor_speeds(0,0))

avg_speed = np.mean([(m["left_speed"] + m["right_speed"]) / 2 for m in thymio_data])
thymio_mms_speed = distance_mm / (len(thymio_data)*Ts)
print(f"thymio speed in mm/s is {thymio_mms_speed}")
print(f"average raw speed is {avg_speed}")
thymio_speed_to_mms = thymio_mms_speed / raw_speed
print(f"to go from raw thymio speed to mm/s speed the conversion factor is {thymio_speed_to_mms}")



thymio speed in mm/s is 28.626609442060087
average raw speed is 78.37768240343348
to go from raw thymio speed to mm/s speed the conversion factor is 0.28626609442060086
